# BERT を自分の分野のテキストで追加で事前学習する（ドメイン適応）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/08_transformers/mlm_domain_adaptation.ipynb)

NLP クックブック 08-03 ／ Transformers の基本操作

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

BERT の事前学習のデータとは書き方の違う分野（ここでは旧仮名遣いの近代文学）のテキストで、マスク言語モデルの学習を続けたい。その分野の文の穴埋めがうまくなるか、分類の精度に効くかも確かめる。

In [1]:
!pip install -q fugashi unidic-lite datasets

In [2]:
import random
from collections import Counter
import numpy as np
import torch
from datasets import load_dataset

random.seed(0); np.random.seed(0); torch.manual_seed(0)

# 青空文庫の作品のテキストを整形したもの (globis-university/aozorabunko-clean、データセットは CC BY 4.0。
# ここで使う作品は「作品著作権フラグ」が「なし」の、著作権の切れたもの)
ds = load_dataset("globis-university/aozorabunko-clean", split="train")
kyu = [ex for ex in ds if ex["meta"]["文字遣い種別"] == "新字旧仮名" and ex["meta"]["作品著作権フラグ"] == "なし"]
shin = [ex for ex in ds if ex["meta"]["文字遣い種別"] == "新字新仮名" and ex["meta"]["作品著作権フラグ"] == "なし"]

# 作者単位で、追加学習に使う作者 (8 割) と評価だけに使う作者 (2 割) に分ける
authors = sorted({ex["meta"]["人物ID"] for ex in kyu})
random.Random(0).shuffle(authors)
test_authors = set(authors[:len(authors) // 5])
kyu_train = [ex for ex in kyu if ex["meta"]["人物ID"] not in test_authors]
kyu_test = [ex for ex in kyu if ex["meta"]["人物ID"] in test_authors]
print(f"旧仮名の作品: 学習用 {len(kyu_train)} 作品 / 評価用 {len(kyu_test)} 作品 (作者 {len(test_authors)} 人)")

def paragraphs(works, min_len=20):
    return [p.strip() for ex in works for p in ex["text"].split("\n") if len(p.strip()) >= min_len]

print(paragraphs(kyu_test)[:2])

旧仮名の作品: 学習用 3171 作品 / 評価用 1344 作品 (作者 72 人)
['突然、すこしおそろしい音がした。それから、どたんばたんといろ〳〵の物音が矢つぎ早にしたかと思ふと、しいんとなつた。', 'はじめにした声は、俊一がその弟を叱りつけたのである。何か気に障つたことがあつたのだらう。弟と向ひ合つてゐて、俊一は突然怒鳴つた。そして、ちよつと仁王様に似たやうな顔付になつたが、その自分の怒つた顔に自信がないために、どこか抜けたところがありはしないかといふ怖れに駆られた、といつた様子でつと立上り、ばり〳〵と襖を押し開けて奥の間の自分の室へ行かうとした。何か重大な国際会議の席上から脱退する悲壮な日本代表のやうな調子であつた。ばりつと風を巻いて室から出ようとする時、壁に吊してあつた瓢箪が落ちた。勿論、俊一は、あつといふ間にその瓢箪を片足あげてふんづけた。気が立つてゐる時は、かういふ品物を一気に粉みぢんにするのは気持のいいものだ。が、あれほど俊一が力を籠めたにも拘らず、瓢箪は潰れなかつた。のみならずつるりと外れて俊一の足をすくつたから、彼はよろよろとして、危くひつくりかへるところであつた。怒り心頭に発した彼は、勿論電光石火の敏速さを以て瓢箪を潰さんともう一度試みるべきであつたし、又試みようとしたのだが、具合の悪いことに瓢箪は彼の足からすこし離れたところに在つた。それで、不覚にも彼は躊躇してしまつたのである。ちよつとの間、彼は足を出したものかやめたものかとぴく〳〵させてゐた。彼は、はたから見てゐて、それがどのくらゐ哀れな可笑しさを唆るものであるかといふことに忽ち気がついた。その上、尚悪いことには、すこしも騒がず坐つたまゝであつた弟の方をちらりと見てしまつたのである。弟は、意地悪さうな目付をしてにやりと笑ひかけてゐた。']


In [3]:
from transformers import AutoTokenizer

name = "tohoku-nlp/bert-base-japanese-v3"   # 東北大学の日本語 BERT (Apache 2.0)
tok = AutoTokenizer.from_pretrained(name)
print(tok.tokenize("さうして私はかう思ふのである。"))   # 旧仮名は細かく分かれる

def chunks(texts, n, L=128):
    """段落を順にトークン化してつなぎ、[CLS] + (L-2) トークン + [SEP] の長さ L の列に切る
    (長い段落で「Token indices sequence length is longer than ...」の警告が出るが、L に切るので問題ない)"""
    out, buf = [], []
    for t in texts:
        buf += tok(t, add_special_tokens=False)["input_ids"]
        while len(buf) >= L - 2 and len(out) < n:
            out.append([tok.cls_token_id] + buf[:L - 2] + [tok.sep_token_id]); buf = buf[L - 2:]
        if len(out) == n:
            break
    return out

train_paras = paragraphs(kyu_train)
random.Random(0).shuffle(train_paras)                    # 作品が偏らないよう段落を混ぜる
N_TRAIN = 40000                                          # 約 500 万トークン
train_chunks = chunks(train_paras, N_TRAIN)

# 評価用: 旧仮名 (学習に使っていない作者)、新仮名の文学作品、Wikipedia の 3 種類を 1000 列ずつ
def sample_paras(works, k=3000):
    p = paragraphs(works); random.Random(0).shuffle(p); return p[:k]
wiki = load_dataset("wikimedia/wikipedia", "20231101.ja", split="train", streaming=True)   # CC BY-SA 3.0 / GFDL
wiki_paras = [p for a in wiki.take(300) for p in a["text"].split("\n") if len(p) >= 20]
eval_chunks = {"旧仮名 (評価用の作者)": chunks(sample_paras(kyu_test), 1000),
               "新仮名の文学作品": chunks(sample_paras(shin), 1000),
               "Wikipedia": chunks(wiki_paras, 1000)}
print("学習:", len(train_chunks), "列", {k: len(v) for k, v in eval_chunks.items()})
print("[UNK] の割合:", f"{sum(c.count(tok.unk_token_id) for c in train_chunks) / (len(train_chunks) * 128):.4%}")

['さ', '##う', 'し', 'て', '私', 'は', 'か', '##う', '思', '##ふ', 'の', 'で', 'ある', '。']
学習: 40000 列 {'旧仮名 (評価用の作者)': 1000, '新仮名の文学作品': 1000, 'Wikipedia': 1000}
[UNK] の割合: 0.0282%


In [4]:
import math
import torch.nn.functional as F
from transformers import AutoModelForMaskedLM

# このモデルは単語単位のマスク (whole word masking) で事前学習されている。MeCab の 1 語が「さ ##う」のように
# 複数のサブワードに分かれたときは、まとめて隠す (「さ」だけ見せて「##う」を当てさせると簡単すぎる)
is_sub = torch.zeros(len(tok), dtype=torch.bool)
is_sub[[i for t, i in tok.get_vocab().items() if t.startswith("##")]] = True

def word_mask(ids, p=0.15, generator=None):
    """語の先頭を確率 p で選び、その語のサブワードをすべて隠す位置にする"""
    start = ~is_sub[ids]
    pick = start & (torch.rand(ids.shape, generator=generator) < p) & (ids != tok.cls_token_id) & (ids != tok.sep_token_id)
    m = pick.clone()
    for t in range(1, ids.shape[1]):
        m[:, t] = torch.where(start[:, t], pick[:, t], m[:, t - 1])   # 「##」のサブワードは直前と同じ
    return m

def mask_fixed(ids, seed=0):
    """評価用に、隠す位置を固定する (隠した位置はすべて [MASK] にする)"""
    ids = torch.tensor(ids)
    m = word_mask(ids, generator=torch.Generator().manual_seed(seed))
    return ids.masked_fill(m, tok.mask_token_id), ids.masked_fill(~m, -100)

eval_sets = {k: mask_fixed(v) for k, v in eval_chunks.items()}

@torch.no_grad()
def mlm_eval(model, data, bs=64):
    """隠したトークンの平均損失 (負の対数尤度)、パープレキシティ、1 位の予測が当たった割合"""
    model.eval()
    inp, labels = data
    loss = correct = n = 0
    for i in range(0, len(inp), bs):
        x, y = inp[i:i + bs].to(model.device), labels[i:i + bs].to(model.device)
        logits = model(input_ids=x).logits[y != -100].float()
        loss += F.cross_entropy(logits, y[y != -100], reduction="sum").item()
        correct += (logits.argmax(-1) == y[y != -100]).sum().item(); n += (y != -100).sum().item()
    return {"損失": round(loss / n, 3), "PPL": round(math.exp(loss / n), 1), "1 位の正解率": round(correct / n, 3)}

model = AutoModelForMaskedLM.from_pretrained(name).to("cuda")   # 学習するので float32 で読み、計算だけ半精度にする
before = {k: mlm_eval(model, d) for k, d in eval_sets.items()}
for k, v in before.items():
    print(f"{k:16}", v)

旧仮名 (評価用の作者)     {'損失': 3.356, 'PPL': 28.7, '1 位の正解率': 0.425}
新仮名の文学作品         {'損失': 3.076, 'PPL': 21.7, '1 位の正解率': 0.464}
Wikipedia        {'損失': 1.711, 'PPL': 5.5, '1 位の正解率': 0.667}


In [5]:
from transformers import pipeline

examples = ["雨が降つて[MASK]る。", "彼は何も言は[MASK]かつた。", "私はかう[MASK]ふのである。", "それは美しい[MASK]であつた。",
            "日本の首都は[MASK]です。", "私は毎朝[MASK]を飲みます。"]   # 最後の 2 つはこの章の「マスクされた単語を BERT に予測させる (fill-mask)」と同じ現代文

def show_fill(model):
    fill = pipeline("fill-mask", model=model, tokenizer=tok, device=model.device)
    for text in examples:
        print(f"{text:16}", "  ".join(f"{r['token_str']}({r['score']:.2f})" for r in fill(text, top_k=4)))

show_fill(model)

雨が降つて[MASK]る。    ゐ(0.30)  来(0.09)  おけ(0.03)  おら(0.02)
彼は何も言は[MASK]かつた。 なか(0.10)  し(0.10)  つ(0.07)  なさ(0.04)
私はかう[MASK]ふのである。 ま(0.05)  は(0.04)  と(0.04)  な(0.03)
それは美しい[MASK]であつた。 もの(0.25)  こと(0.02)  夢(0.02)  姿(0.01)
日本の首都は[MASK]です。  東京(0.75)  京都(0.05)  大阪(0.04)  北京(0.02)
私は毎朝[MASK]を飲みます。 コーヒー(0.26)  酒(0.17)  ビール(0.11)  薬(0.07)


In [6]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments

def collate(examples):
    """追加の事前学習: 語の 15% を隠して当てる。隠す位置は BERT の事前学習と同じく 80% を [MASK]、10% をランダムな語、10% をそのままにする
    (DataCollatorForLanguageModeling の whole_word_mask は fast トークナイザーが必要で、このモデルでは使えない)"""
    ids = torch.tensor([e["input_ids"] for e in examples])
    m = word_mask(ids)
    r = torch.rand(ids.shape)
    inp = ids.masked_fill(m & (r < 0.8), tok.mask_token_id)
    rnd = m & (r >= 0.8) & (r < 0.9)
    inp[rnd] = torch.randint(len(tok), ids.shape)[rnd]
    return {"input_ids": inp, "labels": ids.masked_fill(~m, -100)}

BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
args = TrainingArguments(output_dir="mlm-out", num_train_epochs=1, learning_rate=5e-5, warmup_steps=100, weight_decay=0.01,
                         per_device_train_batch_size=32, bf16=BF16, fp16=not BF16, logging_steps=125,
                         save_strategy="no", report_to="none", seed=0)
trainer = Trainer(model=model, args=args, train_dataset=Dataset.from_dict({"input_ids": train_chunks}),
                  data_collator=collate)
trainer.train()   # 1250 ステップ。T4 で 10 分前後
print(f"GPU メモリの最大使用量: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
trainer.save_model("bert-kyukana"); tok.save_pretrained("bert-kyukana")

Step,Training Loss
125,3.064376
250,2.921588
375,2.889291
500,2.847911
625,2.810749
750,2.783233
875,2.768788
1000,2.739382
1125,2.733612
1250,2.731065


GPU メモリの最大使用量: 4.9 GB


('bert-kyukana/tokenizer_config.json',
 'bert-kyukana/vocab.txt',
 'bert-kyukana/added_tokens.json')

In [7]:
import pandas as pd

after = {k: mlm_eval(model, d) for k, d in eval_sets.items()}
print(pd.concat({"追加学習の前": pd.DataFrame(before).T, "後": pd.DataFrame(after).T}, axis=1).to_string())
show_fill(model)

             追加学習の前                    後              
                 損失   PPL 1 位の正解率     損失   PPL 1 位の正解率
旧仮名 (評価用の作者)  3.356  28.7   0.425  2.928  18.7   0.471
新仮名の文学作品      3.076  21.7   0.464  3.017  20.4   0.455
Wikipedia     1.711   5.5   0.667  1.745   5.7   0.642
雨が降つて[MASK]る。    ゐ(0.23)  来(0.10)  了(0.08)  くれ(0.02)
彼は何も言は[MASK]かつた。 な(0.99)  なか(0.01)  無(0.00)  し(0.00)
私はかう[MASK]ふのである。 笑(0.33)  云(0.12)  言(0.09)  思(0.06)
それは美しい[MASK]であつた。 もの(0.14)  こと(0.04)  夢(0.02)  女(0.02)
日本の首都は[MASK]です。  東京(0.62)  京都(0.09)  大阪(0.04)  日本橋(0.03)
私は毎朝[MASK]を飲みます。 酒(0.49)  茶(0.06)  コーヒー(0.05)  ビール(0.05)


In [8]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, set_seed
from transformers.utils import logging as hf_logging

# 下流のタスク: 旧仮名の作品の段落が、小説 (NDC 913) か随筆・評論 (NDC 914) かを当てる。
# 学習は追加学習に使った作者、評価は評価用の作者の作品から取る (同じ作者や作品が両方に入らない)
def clf_data(works, per_class, seed):
    rng, rows = random.Random(seed), []
    for label, ndc in enumerate(["NDC 913", "NDC 914"]):
        ps = []
        for ex in works:
            if ex["meta"]["分類番号"] == ndc:
                p = paragraphs([ex], 50)
                ps += [(t, ex["meta"]["作品ID"]) for t in rng.sample(p, min(3, len(p)))]   # 1 作品から最大 3 段落
        rng.shuffle(ps)
        rows += [{"text": t, "work": w, "label": label} for t, w in ps[:per_class]]
    enc = tok([r["text"] for r in rows], truncation=True, max_length=128)
    return Dataset.from_dict({k: [r[k] for r in rows] for k in ["text", "work", "label"]} | dict(enc))

hf_logging.set_verbosity_error()   # 読み込みのたびに出る重みの一覧 (分類の層が新しく作られた、など) を表示しない
clf_train, clf_test = clf_data(kyu_train, 500, 0), clf_data(kyu_test, 600, 1)
print("学習", len(clf_train), Counter(clf_train["label"]), "/ 評価", len(clf_test), Counter(clf_test["label"]))

def finetune(path, seed):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2, id2label={0: "小説", 1: "随筆・評論"},
                                                               label2id={"小説": 0, "随筆・評論": 1})
    if path != name:
        # 保存した MLM のモデルには、分類で使う pooler の層がない。条件をそろえるため、pooler は元のモデルのものを使い、
        # エンコーダー (埋め込みと 12 層) の重みだけを追加学習したものに差し替える
        missing = model.bert.load_state_dict(AutoModelForMaskedLM.from_pretrained(path).bert.state_dict(), strict=False).missing_keys
        assert all(k.startswith("pooler") for k in missing), missing
    args = TrainingArguments(output_dir="clf-out", num_train_epochs=3, learning_rate=3e-5, warmup_steps=10, weight_decay=0.01,
                             per_device_train_batch_size=32, per_device_eval_batch_size=128, bf16=BF16, fp16=not BF16,
                             save_strategy="no", logging_strategy="no", report_to="none", seed=seed)
    trainer = Trainer(model=model, args=args, train_dataset=clf_train.remove_columns(["text", "work"]),
                      data_collator=DataCollatorWithPadding(tok))
    trainer.train()
    return trainer.predict(clf_test.remove_columns(["text", "work"])).predictions.argmax(-1)

del model, trainer; torch.cuda.empty_cache()
y = np.array(clf_test["label"])
correct = {}
for label, path in [("元の BERT", name), ("追加学習した BERT", "bert-kyukana")]:   # T4 で 1 回 40 秒程度 × 6
    correct[label] = np.array([finetune(path, seed) == y for seed in range(3)])   # (シード, 評価の段落)
    print(f"{label:12} 正解率", " ".join(f"{a:.3f}" for a in correct[label].mean(1)), f" 平均 {correct[label].mean():.3f}")

学習 1000 Counter({0: 500, 1: 500}) / 評価 1200 Counter({0: 600, 1: 600})
{'train_runtime': '7.288', 'train_samples_per_second': '411.6', 'train_steps_per_second': '13.17', 'train_loss': '0.4459', 'epoch': '3'}
{'train_runtime': '7.18', 'train_samples_per_second': '417.8', 'train_steps_per_second': '13.37', 'train_loss': '0.4187', 'epoch': '3'}
{'train_runtime': '7.172', 'train_samples_per_second': '418.3', 'train_steps_per_second': '13.39', 'train_loss': '0.4227', 'epoch': '3'}
元の BERT      正解率 0.812 0.830 0.829  平均 0.824
{'train_runtime': '7.136', 'train_samples_per_second': '420.4', 'train_steps_per_second': '13.45', 'train_loss': '0.4422', 'epoch': '3'}
{'train_runtime': '7.124', 'train_samples_per_second': '421.1', 'train_steps_per_second': '13.47', 'train_loss': '0.4003', 'epoch': '3'}
{'train_runtime': '7.133', 'train_samples_per_second': '420.6', 'train_steps_per_second': '13.46', 'train_loss': '0.4202', 'epoch': '3'}
追加学習した BERT  正解率 0.829 0.825 0.830  平均 0.828


In [9]:
# 正解率の差のブートストラップ信頼区間: 同じ作品の段落は似ているので、作品を単位にリサンプリングする
works = np.array(clf_test["work"])
uniq = np.unique(works)
idx = {w: np.where(works == w)[0] for w in uniq}
diff = correct["追加学習した BERT"].mean(0) - correct["元の BERT"].mean(0)   # 段落ごとに、3 シードの平均の差
rng = np.random.default_rng(0)
boot = [np.mean(np.concatenate([diff[idx[w]] for w in rng.choice(uniq, len(uniq))])) for _ in range(2000)]
print(f"正解率の差 {diff.mean():+.3f}  95% 信頼区間 [{np.percentile(boot, 2.5):+.3f}, {np.percentile(boot, 97.5):+.3f}]  (作品 {len(uniq)} 件)")

正解率の差 +0.004  95% 信頼区間 [-0.004, +0.012]  (作品 577 件)


- 事前学習済みのモデルを、ラベルのない分野のテキストでマスク言語モデルとして学習し直してから微調整する方法は、ドメイン適応事前学習 (domain-adaptive pretraining) と呼ばれる (Gururangan et al., 2020)。医療・法律・社内文書などでも、テキストを差し替えれば同じ手順で使える。
- この実行（約 500 万トークン、1,250 ステップ）では、学習に使っていない作者の旧仮名の文で、隠した語の損失が 3.356 → 2.928、1 位の正解率が 0.425 → 0.471 に改善した。fill-mask の個々の例はまちまちで、「彼は何も言は[MASK]かつた。」の「な」は 1 位 (0.99) になったが、「雨が降つて[MASK]る。」の「ゐ」は 0.30 → 0.23 と下がった。個々の例よりも、多くの位置で数えた損失や正解率で判断する。
- モデルは CC-100 で 1M ステップ学習したあと Wikipedia（2023 年 1 月の版）でさらに 1M ステップ学習されている（モデルカード）ので、Wikipedia は元の分野にあたる。Wikipedia では損失が 1.711 → 1.745、正解率が 0.667 → 0.642 と少し悪くなり、「私は毎朝[MASK]を飲みます。」の 1 位は「コーヒー」から「酒」に変わった。学習を長く続けたり学習率を大きくしたりすると元の分野の性能が大きく落ちることがあり、破滅的忘却と呼ばれる（機械翻訳の章の「学習済みの翻訳モデルを自分の分野のデータでファインチューニングする」の注意書きも参照）。元の分野のテキストを混ぜる、学習率やステップ数を抑える、といった対策がある（ここでは試していない）。
- 評価の「1 位の正解率」は、この章の「マスクされた単語を BERT に予測させる (fill-mask)」の 1 位の予測が当たる割合を、隠した全トークンで数えたもの。この章の「マスク言語モデルで文の自然さを測る（疑似対数尤度）」は 1 トークンずつ隠して文長の回数だけ推論するが、ここでは語の 15% を同時に隠して 1 回で済ませている。隠す位置を固定しているので前後の比較はできるが、疑似対数尤度の値とは比べられない。
- tohoku-nlp/bert-base-japanese-v3 は、MeCab の 1 語のサブワードをまとめて隠す whole word masking で事前学習されている（モデルカード）。評価と追加学習でもそれにそろえた。`DataCollatorForLanguageModeling(whole_word_mask=True)` は fast トークナイザーの文字位置を使うので、MeCab を使うこのモデルのトークナイザーでは使えず、「##」の付くサブワードを手がかりに自分で書いた。
- 下流の分類（小説か随筆・評論か）では、正解率の平均が 0.824 → 0.828 で、差 +0.004 の 95% 信頼区間（作品 577 件を単位にしたブートストラップ）は [-0.004, +0.012] と 0 をまたいだ。シードによっても 0.01〜0.02 程度動く（元の BERT は 0.812〜0.830）。マスク言語モデルの損失が大きく改善しても、分類の精度が上がるとは限らないので、下流のタスクで、シードを変えた複数回の実行で確かめる。
- 保存した `AutoModelForMaskedLM` には、分類で使う pooler の層が含まれない。そのまま `AutoModelForSequenceClassification` で読むと pooler がランダムに初期化され、元のモデルとの比較が不公平になるので、上のコードでは pooler を元のモデルから取り、エンコーダーだけを差し替えた。
- データは青空文庫（globis-university/aozorabunko-clean、CC BY 4.0。著作権の切れた作品だけを収録）と日本語 Wikipedia（wikimedia/wikipedia、CC BY-SA 3.0 / GFDL）、モデルは東北大学の BERT (Apache 2.0)。数値は A5000 で float16 の経路（`BF16 = False`、T4 と同じ dtype）で実行したもので、GPU メモリの最大使用量は 4.9 GB だった。追加学習したモデルは `bert-kyukana` に保存される。同じランタイムなら、fill-mask の `model=` や疑似対数尤度の `name` に `"bert-kyukana"` を渡せば使える。別のノートブックで使うなら、この章の「モデルを保存・読み込みする、Hub にアップロードする」のように Google ドライブに保存するか Hub に上げる。